# Dam Inspection Commentary — Phase 0 v2: Model Selection Experiment

## Objective
Scientifically compare three Vision-Language Models (VLMs) on dam inspection imagery under zero-shot conditions.
The best-performing model will be selected for domain-specific fine-tuning in Phase 1.

## Candidates
| Model | Parameters | Provider | Key Strength |
|---|---|---|---|
| LLaVA-1.5 7B | 7B | LLaVA team / HuggingFace | Instruction-tuned, strong long-form generation |
| Qwen2-VL 2B Instruct | 2B | Alibaba | Recent (2024), strong visual grounding, very efficient |
| InternVL2 2B | 2B | OpenGVLab | Strong vision-language alignment, instruction-tuned |

## Why These Models
All previous candidates (PaliGemma-mix, Moondream2) were designed for short VQA tasks and
cannot produce long structured reports regardless of prompting. These three candidates are
all instruction-tuned for detailed, multi-section text generation from images.

## Protocol
- Identical prompt for all models
- Same set of proxy dam images
- Each model loaded, evaluated, then explicitly unloaded before the next
- 4-bit quantization (bitsandbytes) for all models to fit T4 16GB VRAM budget
- Outputs saved to JSON for downstream evaluation in Phase 2

## 0. Environment Setup

In [ ]:
# Verify GPU before anything else
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU — switch runtime to T4 GPU before proceeding')

In [ ]:
%%capture
!pip install transformers==4.45.0 accelerate bitsandbytes Pillow requests torch torchvision
!pip install qwen-vl-utils  # required for Qwen2-VL image preprocessing
!pip install rouge-score bert-score nltk sentencepiece  # metrics for Phase 2

In [ ]:
import gc
import json
import time
import torch
import requests
import numpy as np
from PIL import Image
from io import BytesIO
from pathlib import Path
from datetime import datetime
from transformers import BitsAndBytesConfig

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}')
    print(f'VRAM   : {props.total_memory / 1e9:.1f} GB')

## 1. Proxy Dataset

Public domain / CC-licensed dam images used as a proxy until the company dataset arrives.
Images cover the main dam typologies encountered in professional inspection practice:
gravity, arch, embankment, and operational spillway configurations.

> **Swapping in company data:** Replace the entries in `PROXY_IMAGES` with local file paths.
> `load_image()` accepts both URLs and local paths transparently.

In [ ]:
# Format: (url_or_local_path, image_id, dam_type)
# ↓ Replace URLs with your own verified working links or local paths
PROXY_IMAGES = [
    # --- Add your image sources here ---
    # Example local file:
    # ('/content/my_dam_image.jpg', 'site_A_downstream', 'gravity'),
    #
    # Example URL (must be a direct image link, not a webpage):
    # ('https://example.com/dam.jpg', 'example_dam', 'embankment'),
]

def load_image(source: str) -> Image.Image:
    """Load image from URL or local path. Returns PIL RGB Image."""
    if source.startswith('http'):
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(source, headers=headers, timeout=20)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert('RGB')
    return Image.open(source).convert('RGB')

print('Loading proxy dataset...')
dataset = []
for source, img_id, dam_type in PROXY_IMAGES:
    try:
        img = load_image(source)
        dataset.append({
            'id': img_id,
            'dam_type': dam_type,
            'image': img,
            'source': source
        })
        print(f'  ✓ {img_id} ({dam_type}) — {img.size[0]}x{img.size[1]}')
    except Exception as e:
        print(f'  ✗ {img_id} — {e}')

print(f'\nDataset ready: {len(dataset)} images')
if len(dataset) == 0:
    print('WARNING: No images loaded. Add your image sources to PROXY_IMAGES above.')

## 2. Engineering Prompt

A single standardized prompt is used across all three models to ensure a fair comparison.
The prompt follows the **role + task + format + uncertainty** design pattern:
- **Role**: establishes domain expertise context
- **Task**: bullet-format sections (avoids numbered-list continuation loops)
- **Format**: requests formal engineering language
- **Uncertainty**: instructs the model to flag unassessable features rather than hallucinate

In [ ]:
ENGINEERING_PROMPT = """You are a licensed civil engineer specializing in dam safety inspection and structural assessment.
Analyze this dam image and write a detailed professional inspection report covering the following:

- STRUCTURE TYPE: Identify the dam type (gravity, arch, embankment, buttress, rockfill, etc.) and visible construction material.
- STRUCTURAL ELEMENTS: Describe observable components such as crest, downstream face, abutments, spillway, stilling basin, and outlets.
- SURFACE CONDITION: Note any visible cracking, spalling, seepage staining, erosion, settlement, or material deterioration.
- VEGETATION AND ENCROACHMENT: Assess vegetation presence on the embankment or structure body and any nearby encroachments.
- HYDRAULIC OBSERVATIONS: Describe reservoir level, downstream flow conditions, and any signs of overtopping risk or scour.
- ANOMALIES AND CONCERNS: Flag any observable defects, deformations, or conditions requiring further investigation.
- RECOMMENDED ACTIONS: Propose immediate, short-term, and long-term inspection or maintenance actions.

Use formal engineering language throughout. Be precise and specific to what is visible in the image.
If a feature cannot be reliably assessed from the image, state: Not assessable from available imagery.
Do not reference the dam by name or recall facts from memory. Base your report strictly on visual observation."""

print(f'Prompt ready — {len(ENGINEERING_PROMPT.split())} words')

## 3. Shared Utilities

In [ ]:
def clear_gpu_memory():
    """Aggressively free VRAM between model loads."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    time.sleep(2)
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'VRAM after cleanup: {used:.2f} / {total:.1f} GB')


def get_bnb_config() -> BitsAndBytesConfig:
    """Standard 4-bit NF4 quantization config for T4 VRAM budget."""
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )


def save_results(model_name: str, results: list):
    """Persist model results to JSON (image objects excluded)."""
    serializable = [{k: v for k, v in r.items() if k != 'image'} for r in results]
    fname = model_name.replace('/', '_').replace('-', '_') + '_results.json'
    path = OUTPUT_DIR / fname
    with open(path, 'w') as f:
        json.dump({model_name: serializable}, f, indent=2)
    print(f'  Saved → {path}')
    return path


def run_inference_loop(model_name: str, infer_fn, dataset: list) -> list:
    """Generic inference loop — same structure for every model."""
    print(f'\nRunning {model_name} on {len(dataset)} images...')
    results = []
    for sample in dataset:
        try:
            start = time.time()
            commentary = infer_fn(sample['image'], ENGINEERING_PROMPT)
            elapsed = round(time.time() - start, 2)
            results.append({
                'id': sample['id'],
                'dam_type': sample['dam_type'],
                'model': model_name,
                'commentary': commentary,
                'inference_time_s': elapsed,
                'word_count': len(commentary.split()),
            })
            print(f'  ✓ [{sample["id"]}] {elapsed}s — {len(commentary.split())} words')
        except Exception as e:
            print(f'  ✗ [{sample["id"]}] Error: {e}')
            results.append({
                'id': sample['id'],
                'dam_type': sample['dam_type'],
                'model': model_name,
                'commentary': f'ERROR: {e}',
                'inference_time_s': 0,
                'word_count': 0,
            })
    return results


# Shared results store across all model sections
all_results = {}
print('Utilities ready.')

## 4. Model 1 — LLaVA-1.5 7B

**Architecture:** CLIP ViT-L/14@336px vision encoder + Vicuna-7B language model  
**Why included:** LLaVA-1.5 is the most established open-source instruction-tuned VLM for
detailed visual description tasks. It is specifically trained on multi-turn conversations
requiring long structured responses — making it the natural baseline for this task.

**HuggingFace:** `llava-hf/llava-1.5-7b-hf` (no license gating required)

In [ ]:
from transformers import LlavaProcessor, LlavaForConditionalGeneration

LLAVA_MODEL_ID = 'llava-hf/llava-1.5-7b-hf'

print(f'Loading {LLAVA_MODEL_ID} (4-bit quantized)...')
llava_processor = LlavaProcessor.from_pretrained(LLAVA_MODEL_ID)
llava_model = LlavaForConditionalGeneration.from_pretrained(
    LLAVA_MODEL_ID,
    quantization_config=get_bnb_config(),
    device_map='auto',
)
llava_model.eval()
print('LLaVA-1.5 7B loaded.')
clear_gpu_memory()

In [ ]:
def run_llava15(image: Image.Image, prompt: str, max_new_tokens: int = 600) -> str:
    # LLaVA-1.5 requires the USER/ASSISTANT chat template with <image> token
    conversation = f'USER: <image>\n{prompt}\nASSISTANT:'
    inputs = llava_processor(
        text=conversation,
        images=image,
        return_tensors='pt'
    ).to(DEVICE)
    input_len = inputs['input_ids'].shape[-1]

    with torch.inference_mode():
        output = llava_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=4,
        )
    return llava_processor.decode(output[0][input_len:], skip_special_tokens=True).strip()


llava_results = run_inference_loop('llava-1.5-7b', run_llava15, dataset)
all_results['llava-1.5-7b'] = llava_results
save_results('llava-1.5-7b', llava_results)

if llava_results:
    print(f'\n--- Sample output: {llava_results[0]["id"]} ---')
    print(llava_results[0]['commentary'])

In [ ]:
del llava_model, llava_processor
clear_gpu_memory()

## 5. Model 2 — Qwen2-VL 2B Instruct

**Architecture:** ViT with Naive Dynamic Resolution + Qwen2 2B language model  
**Why included:** Released in 2024, Qwen2-VL represents the current generation of efficient VLMs.
Its dynamic resolution mechanism allows it to process images at their native resolution
rather than forcing a fixed crop — an advantage for dam images where structural details
may appear at varying scales.

**HuggingFace:** `Qwen/Qwen2-VL-2B-Instruct` (no license gating required)

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

QWEN_MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'

print(f'Loading {QWEN_MODEL_ID} (4-bit quantized)...')
qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    quantization_config=get_bnb_config(),
    device_map='auto',
)
qwen_model.eval()
print('Qwen2-VL 2B loaded.')
clear_gpu_memory()

In [ ]:
def run_qwen2vl(image: Image.Image, prompt: str, max_new_tokens: int = 600) -> str:
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': prompt},
        ]
    }]
    text = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        return_tensors='pt'
    ).to(DEVICE)
    input_len = inputs['input_ids'].shape[-1]

    with torch.inference_mode():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=4,
        )
    return qwen_processor.decode(output[0][input_len:], skip_special_tokens=True).strip()


qwen_results = run_inference_loop('qwen2-vl-2b', run_qwen2vl, dataset)
all_results['qwen2-vl-2b'] = qwen_results
save_results('qwen2-vl-2b', qwen_results)

if qwen_results:
    print(f'\n--- Sample output: {qwen_results[0]["id"]} ---')
    print(qwen_results[0]['commentary'])

In [ ]:
del qwen_model, qwen_processor
clear_gpu_memory()

## 6. Model 3 — InternVL2 2B

**Architecture:** InternViT-300M vision encoder + InternLM2 1.8B language model  
**Why included:** InternVL2 achieves competitive performance against models 3-4x its size
on dense visual description benchmarks. At 2B parameters it fits comfortably on T4,
making it a strong candidate if efficiency is a deployment priority.

**HuggingFace:** `OpenGVLab/InternVL2-2B` (no license gating required)

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

INTERN_MODEL_ID = 'OpenGVLab/InternVL2-2B'

# InternVL2 requires trust_remote_code and does not use BitsAndBytes
# in the standard way — load in 8-bit via its own mechanism instead
print(f'Loading {INTERN_MODEL_ID}...')
intern_tokenizer = AutoTokenizer.from_pretrained(
    INTERN_MODEL_ID,
    trust_remote_code=True
)
intern_model = AutoModel.from_pretrained(
    INTERN_MODEL_ID,
    torch_dtype=torch.float16,
    load_in_8bit=True,          # 8-bit is more stable than 4-bit for InternVL2
    device_map='auto',
    trust_remote_code=True,
)
intern_model.eval()
print('InternVL2 2B loaded.')
clear_gpu_memory()

In [ ]:
# InternVL2 requires its own image preprocessing pipeline
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
INTERN_IMG_SIZE = 448

def build_intern_transform():
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB')),
        T.Resize((INTERN_IMG_SIZE, INTERN_IMG_SIZE),
                 interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

intern_transform = build_intern_transform()

def run_internvl2(image: Image.Image, prompt: str, max_new_tokens: int = 600) -> str:
    pixel_values = intern_transform(image).unsqueeze(0).to(torch.float16).to(DEVICE)

    generation_config = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=4,
    )
    # InternVL2 uses its own chat() method
    response = intern_model.chat(
        intern_tokenizer,
        pixel_values,
        prompt,
        generation_config,
    )
    return response.strip()


intern_results = run_inference_loop('internvl2-2b', run_internvl2, dataset)
all_results['internvl2-2b'] = intern_results
save_results('internvl2-2b', intern_results)

if intern_results:
    print(f'\n--- Sample output: {intern_results[0]["id"]} ---')
    print(intern_results[0]['commentary'])

In [ ]:
del intern_model, intern_tokenizer
clear_gpu_memory()

## 7. Quantitative Comparison

Four proxy metrics are computed. These are **structural and linguistic proxies** —
they measure output characteristics, not engineering correctness.
The full domain-specific rubric is applied in Phase 2.

| Metric | What it measures | Ideal direction |
|---|---|---|
| Avg. word count | Output verbosity | Higher (more complete) |
| Section coverage | % of 7 required sections present | Higher |
| Technical term density | Domain vocab per 100 words | Higher |
| Avg. inference time | Deployment feasibility | Lower |

In [ ]:
# Load all saved results — works even if models were run in separate Colab sessions
RESULT_FILES = {
    'llava-1.5-7b'  : 'llava_1.5_7b_results.json',
    'qwen2-vl-2b'   : 'qwen2_vl_2b_results.json',
    'internvl2-2b'  : 'internvl2_2b_results.json',
}

loaded_results = {}
for model_name, fname in RESULT_FILES.items():
    # Try both the explicit filename and the auto-generated name from save_results()
    candidates = [
        OUTPUT_DIR / fname,
        OUTPUT_DIR / (model_name.replace('/', '_').replace('-', '_') + '_results.json')
    ]
    for fpath in candidates:
        if fpath.exists():
            with open(fpath) as f:
                data = json.load(f)
            loaded_results[model_name] = list(data.values())[0]
            print(f'  ✓ Loaded {model_name} ({len(loaded_results[model_name])} samples)')
            break
    else:
        print(f'  ✗ {model_name} — results not found, run its cell above first')

print(f'\nModels available for comparison: {list(loaded_results.keys())}')

In [ ]:
# Engineering domain vocabulary
ENGINEERING_TERMS = {
    # Dam typology
    'gravity', 'arch', 'embankment', 'buttress', 'rockfill', 'earthfill',
    # Structural components
    'crest', 'abutment', 'spillway', 'stilling', 'basin', 'outlet', 'parapet',
    'downstream', 'upstream', 'foundation', 'grouting', 'drainage', 'weir', 'toe',
    'apron', 'cutoff', 'gallery', 'penstock',
    # Observed defects
    'cracking', 'spalling', 'seepage', 'erosion', 'settlement', 'subsidence',
    'scour', 'piping', 'overtopping', 'deformation', 'deterioration', 'leakage',
    'saturation', 'disintegration', 'displacement',
    # Inspection / assessment vocabulary
    'inspection', 'assessment', 'monitoring', 'remediation', 'instrumentation',
    'structural', 'geotechnical', 'hydraulic', 'investigate', 'remediate',
    'rehabilitate', 'maintenance', 'intervention',
}

# Keywords expected in each of the 7 required sections
SECTION_KEYWORDS = [
    ['structure type', 'dam type', 'construction material', 'gravity', 'arch', 'embankment'],
    ['structural element', 'crest', 'spillway', 'abutment', 'downstream face'],
    ['surface condition', 'cracking', 'spalling', 'seepage', 'erosion', 'deterioration'],
    ['vegetation', 'encroachment', 'plant', 'growth', 'tree'],
    ['hydraulic', 'reservoir', 'flow', 'water level', 'overtopping'],
    ['anomal', 'concern', 'defect', 'issue', 'deform', 'flag'],
    ['recommend', 'action', 'maintenance', 'investigate', 'monitor'],
]

def score_section_coverage(text: str) -> float:
    """Fraction of the 7 required sections present (keyword match)."""
    text_lower = text.lower()
    found = sum(
        1 for kw_group in SECTION_KEYWORDS
        if any(kw in text_lower for kw in kw_group)
    )
    return round(found / len(SECTION_KEYWORDS), 3)

def score_technical_density(text: str) -> float:
    """Engineering term occurrences per 100 words."""
    words = text.lower().split()
    if not words:
        return 0.0
    hits = sum(1 for w in words if w.strip('.,;:()[]') in ENGINEERING_TERMS)
    return round((hits / len(words)) * 100, 2)


summary = {}
for model_name, outputs in loaded_results.items():
    valid = [o for o in outputs if not o['commentary'].startswith('ERROR')]
    if not valid:
        continue
    summary[model_name] = {
        'n_samples'              : len(valid),
        'avg_word_count'         : round(np.mean([o['word_count'] for o in valid]), 1),
        'avg_inference_time_s'   : round(np.mean([o['inference_time_s'] for o in valid]), 2),
        'avg_section_coverage'   : round(np.mean([score_section_coverage(o['commentary']) for o in valid]), 3),
        'avg_technical_density'  : round(np.mean([score_technical_density(o['commentary']) for o in valid]), 2),
    }

# Pretty-print comparison table
models = list(summary.keys())
metrics = ['n_samples', 'avg_word_count', 'avg_inference_time_s',
           'avg_section_coverage', 'avg_technical_density']

col_w = 22
print('\n' + '='*80)
print('PHASE 0 — QUANTITATIVE COMPARISON')
print('='*80)
print(f'{"Metric":<30}' + ''.join(f'{m:<{col_w}}' for m in models))
print('-'*80)
for metric in metrics:
    row = f'{metric:<30}'
    for m in models:
        row += f'{str(summary[m].get(metric, "N/A")):<{col_w}}'
    print(row)
print('='*80)

with open(OUTPUT_DIR / 'phase0_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSummary saved → outputs/phase0_summary.json')

## 8. Qualitative Side-by-Side Inspection

Quantitative metrics are proxies. This cell prints the actual outputs for the
same image across all models — the qualitative comparison that complements the table above
and is the most informative signal for model selection.

In [ ]:
def compare_outputs(image_id: str):
    """Print all model outputs for a given image."""
    print(f'\n{"="*80}')
    print(f'IMAGE ID: {image_id}')
    print(f'{"="*80}')
    for model_name, outputs in loaded_results.items():
        entry = next((o for o in outputs if o['id'] == image_id), None)
        if entry:
            stats = f'{entry["word_count"]} words | {entry["inference_time_s"]}s'
            sec = score_section_coverage(entry['commentary'])
            tec = score_technical_density(entry['commentary'])
            print(f'\n[ {model_name.upper()} — {stats} | coverage={sec} | tech_density={tec} ]')
            print(entry['commentary'])
        else:
            print(f'\n[ {model_name.upper()} — no output ]')


# Compare all images — comment out the loop and use a single call to focus on one
if dataset:
    for sample in dataset:
        compare_outputs(sample['id'])
else:
    # If dataset was not reloaded in this session, use IDs from saved results
    ids = list({o['id'] for outputs in loaded_results.values() for o in outputs})
    for img_id in ids:
        compare_outputs(img_id)

## 9. Model Selection Decision

Complete this cell after reviewing Sections 7 and 8.
This documented decision becomes the model selection section of your project report.

In [ ]:
# ── Fill in after reviewing results ───────────────────────────────────────────
SELECTED_MODEL = ''  # e.g. 'llava-1.5-7b'

SELECTION_RATIONALE = """
[Complete after Phase 0 evaluation]

Suggested structure:

<Model X> was selected for fine-tuning based on the following criteria evaluated
across <N> proxy dam images under zero-shot conditions:

Quantitative:
- Highest section coverage: X/7 required sections present on average (vs Y for competitors)
- Highest technical term density: X per 100 words
- Acceptable inference time: Xs average on T4 GPU

Qualitative:
- Outputs demonstrated accurate identification of dam typology
- Structural element descriptions were grounded in visible image features
- [Note any specific strengths observed]

Architecture considerations for fine-tuning:
- [Note why this architecture is suitable for QLoRA fine-tuning]
- [Note HuggingFace PEFT compatibility]
"""
# ──────────────────────────────────────────────────────────────────────────────

decision = {
    'selected_model'  : SELECTED_MODEL,
    'rationale'       : SELECTION_RATIONALE,
    'timestamp'       : datetime.now().isoformat(),
    'phase0_summary'  : summary,
}

with open(OUTPUT_DIR / 'model_selection_decision.json', 'w') as f:
    json.dump(decision, f, indent=2)

print(f'Selected model : {SELECTED_MODEL or "(not yet filled in)"}')
print(f'Decision saved → outputs/model_selection_decision.json')
if SELECTED_MODEL:
    print('\n→ Proceed to Phase 1: QLoRA Fine-tuning Pipeline')

---
## Phase 0 Complete

**Outputs produced:**
```
outputs/
├── llava_1_5_7b_results.json          ← LLaVA-1.5 inference outputs
├── qwen2_vl_2b_results.json           ← Qwen2-VL inference outputs
├── internvl2_2b_results.json          ← InternVL2 inference outputs
├── phase0_summary.json                ← Comparative metrics table
└── model_selection_decision.json      ← Documented model selection
```

**Next:** Phase 1 — QLoRA Fine-tuning Pipeline on the selected model with company dataset.